Create a product that can generate marketing brochers about a company
- For prospective clients
- For incvestors
- For Recruitors

Use Technology
- Local OpenAI API
- Use one shot prompting
- Steam back result and show with formatting

In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents, fetch_website_links
from IPython.display import Markdown, display, update_display

In [20]:
load_dotenv(override=True)
client = OpenAI()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("OPENAI_API_KEY environment variable not set")
else:
    print("OPENAI_API_KEY environment variable is set")

MODEL = 'gpt-5-nano'
openai = OpenAI()
ollama = OpenAI(base_url="http://127.0.0.1:11434/v1")
OLLAMA_MODEL = 'gemma4:latest'

OPENAI_API_KEY environment variable is set


In [8]:
links = fetch_website_links("https://prakrutikspalon.com/")
links

['https://prakrutikspalon.com/user/login',
 '#',
 'https://prakrutikspalon.com',
 'https://prakrutikspalon.com/user/login',
 '#',
 'https://prakrutikspalon.com',
 'https://prakrutikspalon.com/combos',
 'https://prakrutikspalon.com/category/summer-care',
 'https://prakrutikspalon.com/category/miniatures',
 'https://prakrutikspalon.com/category/hair',
 '#',
 'https://prakrutikspalon.com/category/concern/hairfall-control',
 'https://prakrutikspalon.com/category/concern/rough-dry-frizzy-hair',
 'https://prakrutikspalon.com/category/concern/chemically-damaged-hair',
 'https://prakrutikspalon.com/category/concern/hair-nourishment',
 'https://prakrutikspalon.com/category/concern/itchy-scalp',
 'https://prakrutikspalon.com/category/concern/hair-color',
 'https://prakrutikspalon.com/category/concern/dandruff',
 'https://prakrutikspalon.com/category/sub-category/hair/curly-hair',
 'https://prakrutikspalon.com/category/sub-category/hair/dandruff-solution',
 'https://prakrutikspalon.com/category/s

In [10]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [11]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [13]:
print(get_links_user_prompt("https://prakrutikspalon.com/"))


Here is the list of links on the website https://prakrutikspalon.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://prakrutikspalon.com/user/login
#
https://prakrutikspalon.com
https://prakrutikspalon.com/user/login
#
https://prakrutikspalon.com
https://prakrutikspalon.com/combos
https://prakrutikspalon.com/category/summer-care
https://prakrutikspalon.com/category/miniatures
https://prakrutikspalon.com/category/hair
#
https://prakrutikspalon.com/category/concern/hairfall-control
https://prakrutikspalon.com/category/concern/rough-dry-frizzy-hair
https://prakrutikspalon.com/category/concern/chemically-damaged-hair
https://prakrutikspalon.com/category/concern/hair-nourishment
https://prakrutikspalon.com/category/concern/itchy-scalp
https://prakrutikspalon.com/category/concern/hair-color
https://pr

In [26]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [27]:
select_relevant_links("https://prakrutikspalon.com/")

{'links': [{'type': 'company page', 'url': 'https://prakrutikspalon.com'},
  {'type': 'contact page', 'url': 'https://prakrutikspalon.com/contact-us'},
  {'type': 'blog page', 'url': 'https://prakrutikspalon.com/blogs'},
  {'type': 'social media - Facebook',
   'url': 'https://www.facebook.com/prakrutikspalon'},
  {'type': 'social media - Instagram',
   'url': 'https://www.instagram.com/prakrutikspalon/'},
  {'type': 'social media - YouTube',
   'url': 'https://www.youtube.com/channel/UCFpt9bRoqkBZzEKrWFVLaBg/featured'}]}

Second Step: Make the broucher!

In [28]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
print(fetch_page_and_all_relevant_links("https://prakrutikspalon.com/"))

## Landing Page:

Prakrutik Spalon | Natural Skincare & Haircare Inspired by Mother Nature

0
0
Home
Combos
Summer Care
Miniatures
Hair
Shop By Concerns
Hairfall Control
Rough / Dry / Frizzy Hair
Chemically Damaged Hair
Hair Nourishment
Itchy Scalp
Hair Color
Dandruff
Curly Hair
Dandruff Solution
Essential Oils
Hair Perfume
Head Massager
Natural Hair Color
Oils & Treatment
Pre Wash Conditioner
Pre Wash Hair Balm
Shampoo
Face
Shop By Concerns
Blackheads / Whiteheads/ Dead & Dry skin removal
Pigmentation
Delay Ageing
Tanned Skin
Active Pimples
Pimple Marks
Dehydrated Skin
Relief from Itching / Rashes
Dry Skin
Oily Skin
Brightening & Radiance
Dark Circles
Active Pimples
Essential Oils
Face Gels
Face Mist
Face Oils
Face Wash
Moisturizers
Night Care
Pimple Marks
Scrubs & Packs (Mini Facial)
Serum
Sun Protection
Mens
De Tanning
Face Wash
Moisturizers
Night Care
Pigmentation
Sun Protection
Aroma
Aromatherapy Freshners
Aromatic Candles
Bath Salts
Body Oils
Closet Freshners
Essential Oils
Hair 

In [46]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
You also need to translate the brochure in Hindi language.
"""


In [47]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
Translate the following information in Hindi.
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [32]:
get_brochure_user_prompt("Prakrutik's Salon", "https://prakrutikspalon.com/")

"\nYou are looking at a company called: Prakrutik's Salon\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nPrakrutik Spalon | Natural Skincare & Haircare Inspired by Mother Nature\n\n0\n0\nHome\nCombos\nSummer Care\nMiniatures\nHair\nShop By Concerns\nHairfall Control\nRough / Dry / Frizzy Hair\nChemically Damaged Hair\nHair Nourishment\nItchy Scalp\nHair Color\nDandruff\nCurly Hair\nDandruff Solution\nEssential Oils\nHair Perfume\nHead Massager\nNatural Hair Color\nOils & Treatment\nPre Wash Conditioner\nPre Wash Hair Balm\nShampoo\nFace\nShop By Concerns\nBlackheads / Whiteheads/ Dead & Dry skin removal\nPigmentation\nDelay Ageing\nTanned Skin\nActive Pimples\nPimple Marks\nDehydrated Skin\nRelief from Itching / Rashes\nDry Skin\nOily Skin\nBrightening & Radiance\nDark Circles\nActive Pimples\nEssential Oils\nFace Gels\nFace Mist\nFace Oils\nFac

In [43]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [34]:
create_brochure("Prakrutik's Salon", "https://prakrutikspalon.com/")

# Prakrutik's Salon  
*Natural Skincare & Haircare Inspired by Mother Nature*

---

## About Us  
Prakrutik's Salon is dedicated to offering premium natural skincare and haircare products inspired by the gifts of Mother Nature. Our philosophy centers on harnessing the power of natural ingredients to promote healthy, vibrant skin and hair without harsh chemicals or synthetic additives.

With a wide range of products carefully formulated to address various concerns, Prakrutik's Salon brings the best of Ayurveda and nature to your daily beauty routine.

---

## Our Product Range  

### Haircare  
- Hairfall Control  
- Treatments for Rough, Dry, Frizzy, and Chemically Damaged Hair  
- Nourishment Oils and Treatments  
- Solutions for Itchy Scalp and Dandruff  
- Natural Hair Colors and Hair Perfumes  
- Pre Wash Conditioners, Balms, Shampoos  
- Head Massagers for scalp relaxation  

### Skincare  
- Solutions for Blackheads, Whiteheads, and Dead & Dry Skin Removal  
- Pigmentation & Spot Lightening  
- Anti-ageing & Brightening products  
- Treatments for Tanned Skin, Pimples, and Pimple Marks  
- Relief for Dry, Oily, Dehydrated, and Irritated Skin  
- Face Gels, Oils, Mists, Face Wash, Moisturizers, Serums, Scrubs & Packs  
- Sun Protection essentials  

### Men’s Care  
- De-tanning  
- Face Wash, Moisturizers, Night Care  
- Pigmentation & Sun Protection  

### Aroma & Wellness  
- Aromatherapy Fresheners, Aromatic Candles, Bath Salts  
- Body Oils, Closet Fresheners, Shoe Rack Fresheners  
- Solid Perfumes and Hair Perfumes  

### Body Care  
- Body Cleansers, Body Oils, Body Polishers  
- Mind Relaxing products  

### Kids’ Care  
- Aloe Vera Gel, Shampoo, Body Cleansers & Oils, Hair Oil, Tooth Care  

### Makeup  
- BB Cream, Cultural Makeup Essentials, Kajal, Lipsticks, Loose Powder, Makeup Cleanser  
- Lip Butters, Lip Lightening Serum  

### Gift Sets  
- Thoughtful gifting options available  

---

## Feature Products  
- **Panchadashaamrut Palmarosa Hair Food** — Controls breakage & hairfall, strengthens follicles  
- **Panch Rakshana Cucumber Sunscreen** — Gel-based, lightweight, non-greasy UVA / UVB protection  
- **Manjistha Spot Lightening Cream** — Lightens pigmentation, brightens, evens skin tone  
- **Ashtaamrut Face Wash** — Brightens skin, evens tone, packed with antioxidants  

---

## Our Culture  
At Prakrutik's Salon, we are passionate about natural beauty and wellness. We believe in sustainability, purity, and the power of nature to heal and enhance. Our team is committed to continuous innovation in natural formulations while being environmentally responsible. We foster a culture of care, respect, and empowerment — for our employees, customers, and the planet.

---

## Careers  
We welcome passionate individuals who share our vision of natural beauty and skincare to join our team. Opportunities range across product development, customer service, marketing, and retail. If you are driven by a desire to make a difference in natural wellness, you will find a meaningful career at Prakrutik’s Salon.

---

## Our Customers  
Prakrutik's Salon serves discerning customers looking for authentic, natural, and effective skincare and haircare solutions. Our clientele includes individuals and families who prioritize health, sustainability, and wellness in their beauty routines. We cater to all ages, skin types, and hair needs with specially tailored products, including men’s and kids’ personal care.

---

## Contact Us  
Explore our complete range and join the natural beauty revolution at [Prakrutik's Salon website](#).

---

**Prakrutik's Salon** — Embrace the beauty of nature every day!

In [44]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Prakrutik's Salon", "https://prakrutikspalon.com/")